In [1]:
from pathlib import Path
import mlflow

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

print(f"MLflow version: {mlflow.__version__}")
print(f"Project folder: {project_root}")

MLflow version: 3.18.0
Project folder: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform


/Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tracking_folder = project_root / "mlflow_tracking"
tracking_folder.mkdir(parents=True, exist_ok=True)

tracking_database = tracking_folder / "mlflow.db"
tracking_uri = f"sqlite:///{tracking_database.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)

print(f"Tracking location: {mlflow.get_tracking_uri()}")

Tracking location: sqlite:////Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/mlflow_tracking/mlflow.db


In [3]:
artifact_folder = tracking_folder / "artifacts"
artifact_folder.mkdir(parents=True, exist_ok=True)

experiment_name = "customer_loyalty_redemption"

existing_experiment = mlflow.get_experiment_by_name(experiment_name)

if existing_experiment is None:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        artifact_location=artifact_folder.as_uri()
    )
else:
    experiment_id = existing_experiment.experiment_id

experiment = mlflow.set_experiment(experiment_id=experiment_id)

print(f"Experiment: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact location: {experiment.artifact_location}")

2026/11/16 13:19:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/16 13:19:03 INFO mlflow.store.db.utils: Updating database tables


Experiment: customer_loyalty_redemption
Experiment ID: 1
Artifact location: file:///Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/mlflow_tracking/artifacts


In [4]:
import json
import pandas as pd

validation_results = pd.read_csv(
    project_root / "reports" / "model_evaluation"
    / "model_comparison_validation.csv"
)

with (project_root / "models" / "modeling_config.json").open(
    encoding="utf-8"
) as file:
    modeling_config = json.load(file)

validation_results.round(4)

,model,roc_auc,average_precision,brier_score
0,Numerical baseline,0.6662,0.2145,0.1186
1,With campaign type,0.7409,0.2832,0.1138
2,XGBoost with campaign type,0.7059,0.2601,0.1149


In [5]:
import joblib

baseline_path = (
    project_root / "models" / "logistic_numeric_baseline.joblib"
)
baseline_model = joblib.load(baseline_path)

baseline_result = validation_results.loc[
    validation_results["model"] == "Numerical baseline"
].iloc[0]

with mlflow.start_run(run_name="logistic_numeric_baseline") as run:
    mlflow.set_tags({
        "logging_mode": "retrospective",
        "model_family": "logistic_regression",
        "campaign_type_included": "false"
    })

    mlflow.log_params(
        baseline_model.named_steps["model"].get_params()
    )

    mlflow.log_metrics({
        "validation_roc_auc": float(baseline_result["roc_auc"]),
        "validation_average_precision": float(
            baseline_result["average_precision"]
        ),
        "validation_brier_score": float(baseline_result["brier_score"])
    })

    mlflow.log_dict(
        {"input_features": modeling_config["numeric_features"]},
        "input_features.json"
    )

    mlflow.log_artifact(str(baseline_path), artifact_path="model_files")
    mlflow.log_artifact(
        str(project_root / "models" / "modeling_config.json"),
        artifact_path="configuration"
    )

    numeric_baseline_run_id = run.info.run_id

print(f"Logged numerical baseline. Run ID: {numeric_baseline_run_id}")

Logged numerical baseline. Run ID: 2ad68ac37b5d4e23bbcab853b3b4b849


In [6]:
selected_path = (
    project_root / "models" / "logistic_with_campaign_type.joblib"
)
selected_model = joblib.load(selected_path)

selected_result = validation_results.loc[
    validation_results["model"] == "With campaign type"
].iloc[0]

with mlflow.start_run(run_name="logistic_with_campaign_type") as run:
    mlflow.set_tags({
        "logging_mode": "retrospective",
        "model_family": "logistic_regression",
        "campaign_type_included": "true",
        "selected_for_final_evaluation": "true",
        "selection_basis": "validation_metrics"
    })

    mlflow.log_params(
        selected_model.named_steps["model"].get_params()
    )

    mlflow.log_metrics({
        "validation_roc_auc": float(selected_result["roc_auc"]),
        "validation_average_precision": float(
            selected_result["average_precision"]
        ),
        "validation_brier_score": float(selected_result["brier_score"])
    })

    mlflow.log_dict(
        {"input_features": modeling_config["campaign_type_features"]},
        "input_features.json"
    )

    mlflow.log_artifact(str(selected_path), artifact_path="model_files")
    mlflow.log_artifact(
        str(project_root / "models" / "modeling_config.json"),
        artifact_path="configuration"
    )

    selected_run_id = run.info.run_id

print(f"Logged selected model. Run ID: {selected_run_id}")

Logged selected model. Run ID: ddd8bfd34b5b4a0d858a2cc5b0b5b359


In [7]:
xgb_path = (
    project_root / "models" / "xgboost_with_campaign_type.joblib"
)
xgb_model = joblib.load(xgb_path)

xgb_result = validation_results.loc[
    validation_results["model"] == "XGBoost with campaign type"
].iloc[0]

with mlflow.start_run(run_name="xgboost_with_campaign_type") as run:
    mlflow.set_tags({
        "logging_mode": "retrospective",
        "model_family": "xgboost",
        "campaign_type_included": "true",
        "selected_for_final_evaluation": "false"
    })

    mlflow.log_params(
        xgb_model.named_steps["model"].get_params()
    )

    mlflow.log_metrics({
        "validation_roc_auc": float(xgb_result["roc_auc"]),
        "validation_average_precision": float(
            xgb_result["average_precision"]
        ),
        "validation_brier_score": float(xgb_result["brier_score"])
    })

    mlflow.log_dict(
        {"input_features": modeling_config["campaign_type_features"]},
        "input_features.json"
    )

    mlflow.log_artifact(str(xgb_path), artifact_path="model_files")
    mlflow.log_artifact(
        str(project_root / "models" / "modeling_config.json"),
        artifact_path="configuration"
    )

    xgb_run_id = run.info.run_id

print(f"Logged XGBoost candidate. Run ID: {xgb_run_id}")

Logged XGBoost candidate. Run ID: 073dde670f5a4c9ea1efee1e8ccef4ec


In [8]:
evaluation_folder = project_root / "reports" / "model_evaluation"

final_metrics_path = evaluation_folder / "final_test_metrics.csv"
final_metrics = pd.read_csv(final_metrics_path).iloc[0]

with mlflow.start_run(run_id=selected_run_id):
    mlflow.log_metrics({
        "test_redemption_rate": float(final_metrics["test_redemption_rate"]),
        "test_roc_auc": float(final_metrics["roc_auc"]),
        "test_average_precision": float(final_metrics["average_precision"]),
        "test_brier_score": float(final_metrics["brier_score"]),
        "test_constant_baseline_brier": float(
            final_metrics["constant_baseline_brier"]
        )
    })

    mlflow.log_artifact(
        str(final_metrics_path),
        artifact_path="final_evaluation"
    )

    mlflow.log_artifact(
        str(evaluation_folder / "final_test_predictions.csv"),
        artifact_path="final_evaluation"
    )

    mlflow.set_tag("final_test_evaluated", "true")

print(f"Attached final test results to run: {selected_run_id}")

Attached final test results to run: ddd8bfd34b5b4a0d858a2cc5b0b5b359


In [9]:
logged_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["attributes.start_time ASC"]
)

print(f"Recorded runs: {len(logged_runs)}")

logged_runs[
    [
        "tags.mlflow.runName",
        "status",
        "metrics.validation_roc_auc",
        "metrics.validation_average_precision",
        "metrics.validation_brier_score",
        "metrics.test_roc_auc",
        "metrics.test_average_precision",
        "metrics.test_brier_score"
    ]
].round(4)

Recorded runs: 3


,tags.mlflow.runName,status,metrics.validation_roc_auc,metrics.validation_average_precision,metrics.validation_brier_score,metrics.test_roc_auc,metrics.test_average_precision,metrics.test_brier_score
0,logistic_numeric_baseline,FINISHED,0.6662,0.2145,0.1186,NaN,NaN,NaN
1,logistic_with_campaign_type,FINISHED,0.7409,0.2832,0.1138,0.6804,0.2579,0.1191
2,xgboost_with_campaign_type,FINISHED,0.7059,0.2601,0.1149,NaN,NaN,NaN


In [10]:
tracking_references = {
    "experiment_name": experiment_name,
    "experiment_id": experiment_id,
    "tracking_uri": mlflow.get_tracking_uri(),
    "runs": {
        "numerical_baseline": numeric_baseline_run_id,
        "selected_logistic_model": selected_run_id,
        "xgboost_candidate": xgb_run_id
    }
}

references_path = project_root / "models" / "mlflow_run_ids.json"

with references_path.open("w", encoding="utf-8") as file:
    json.dump(tracking_references, file, indent=2)

print(f"Saved tracking references: {references_path}")

Saved tracking references: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/models/mlflow_run_ids.json
